In [ ]:
#Question 1
def summarise(amounts, threshold):
    above_threshold = [x for x in amounts if x > threshold]
    above_threshold.sort(reverse=True)

    mean = sum(amounts) / len(amounts)#Calculating mean

    return {
        "above_threshold": above_threshold,
        "mean": mean,
        "count_above": len(above_threshold)
    }

# Call with threshold = 500
result = summarise(amounts, 500)
print(result)

In [ ]:
#Question 2
import pandas as pd
import numpy as np

amounts = pd.read_csv("transactions.csv")["amount"].to_numpy()#Load CSV

print("Mean:", np.mean(amounts))
print("Median:", np.median(amounts))
print("Std:", np.std(amounts))
print("10th percentile:", np.percentile(amounts, 10))
print("90th percentile:", np.percentile(amounts, 90))

outlier_mask = np.abs(amounts - np.mean(amounts)) > 2 * np.std(amounts)

amounts_norm = (amounts - amounts.min()) / (amounts.max() - amounts.min())

counts, bin_edges = np.histogram(amounts_norm, bins=10)

print("Outlier mask:", outlier_mask)
print("Bin edges:", bin_edges)
print("Counts:", counts)

In [ ]:
#Question 3
import pandas as pd

#Load CSV
df = pd.read_csv("smartswipe_transactions.csv")

#Shape, dtypes, missing values
print("Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

#Parse date and create month column
df["date"] = pd.to_datetime(df["date"])

df["month"] = df["date"].dt.month

print("\nMonth column added successfully.")

#Statistical summary for numeric columns
print("\nNumeric Summary:")
print(df.describe())

#Unique city and category values
print("\nUnique Cities:")
print(df["city"].unique())

print("\nNumber of Unique Cities:")
print(df["city"].nunique())

print("\nUnique Categories:")
print(df["category"].unique())

print("\nNumber of Unique Categories:")
print(df["category"].nunique())

In [ ]:
#Question 4
import pandas as pd

#Load data
df = pd.read_csv("smartswipe_transactions.csv")

#Approved transactions only
approved_df = df[df["status"] == "approved"]

percentage = (len(approved_df) / len(df)) * 100

print("Approved Transaction Percentage:")
print(f"{percentage:.2f}%")

#Cashback rate and suspicious flag
df["cashback_rate"] = df["cashback"] / df["amount"]

df["suspicious"] = df["cashback_rate"] > 0.10

print("\nSuspicious Transactions:")
print(df["suspicious"].value_counts())

#Fill missing cashback values with median cashback
median_cashback = df["cashback"].median()

df["cashback"] = df["cashback"].fillna(median_cashback)

print("\nMissing cashback values after filling:")
print(df["cashback"].isnull().sum())

#Spend tier column
bins = [0, 500, 2000, float("inf")]
labels = ["low", "mid", "high"]

df["spend_tier"] = pd.cut(
    df["amount"],
    bins=bins,
    labels=labels,
    right=False
)

print("\nSpend Tier Counts:")
print(df["spend_tier"].value_counts())

In [ ]:
#Question 5
import pandas as pd

df = pd.read_csv("smartswipe_transactions.csv")
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
#Category Statistics
category_stats = (
    df.groupby("category")
      .agg(
          total_spend=("amount", "sum"),
          transaction_count=("amount", "count"),
          avg_transaction_size=("amount", "mean"),
          decline_rate=("status",
                        lambda x: (x == "declined").mean())
      )
      .reset_index()
      .sort_values("total_spend", ascending=False)
)

print(category_stats)
#Top-3 Merchants by Spend Within Each City
merchant_spend = (
    df.groupby(["city", "merchant"])["amount"]
      .sum()
      .reset_index(name="total_spend")
)

top3_merchants = (
    merchant_spend
    .sort_values(
        ["city", "total_spend"],
        ascending=[True, False]
    )
    .groupby("city")
    .head(3)
    .reset_index(drop=True)
)

print(top3_merchants)
#Month × Category Pivot Table
pivot_table = pd.pivot_table(
    df,
    values="amount",
    index="month",
    columns="category",
    aggfunc="mean"
)

print(pivot_table)

In [ ]:
#Question 6
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("smartswipe_transactions.csv")

df["date"] = pd.to_datetime(df["date"])

fig, axes = plt.subplots(3, 1, figsize=(10, 15))

#Bar Chart
category_spend = (
    df.groupby("category")["amount"]
      .sum()
      .sort_values(ascending=False)
)

axes[0].bar(category_spend.index,
            category_spend.values)

axes[0].set_title("Total Spend per Category")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Total Spend")

#Line Chart
daily_spend = (
    df.groupby("date")["amount"]
      .sum()
)

axes[1].plot(
    daily_spend.index,
    daily_spend.values
)

axes[1].set_title("Daily Total Spend")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Total Spend")

axes[1].tick_params(axis="x", rotation=45)

#Histogram
approved = df[df["status"] == "approved"]["amount"]

declined = df[df["status"] == "declined"]["amount"]

axes[2].hist(
    approved,
    bins=20,
    alpha=0.6,
    label="Approved"
)

axes[2].hist(
    declined,
    bins=20,
    alpha=0.6,
    label="Declined"
)

axes[2].set_title(
    "Distribution of Transaction Amounts"
)

axes[2].set_xlabel("Amount")
axes[2].set_ylabel("Frequency")

axes[2].legend()

plt.tight_layout()
plt.show()

In [1]:
#Question 7
import pandas as pd
import numpy as np
#Velocity Fraud Function
def flag_velocity_fraud(df, window='1h', min_txns=3):

    temp = df.copy()

    # Create datetime column if date and time are separate
    if 'datetime' not in temp.columns:
        temp['datetime'] = pd.to_datetime(
            temp['date'].astype(str) + ' ' + temp['time'].astype(str)
        )

    original_index = temp.index

    temp = temp.sort_values(['user_id', 'datetime'])

    def detect(group):
        counts = (
            group.set_index('datetime')
                 .rolling(window)
                 .user_id
                 .count()
        )

        return counts >= min_txns

    fraud_flags = (
        temp.groupby('user_id', group_keys=False)
            .apply(detect)
    )

    temp['velocity_fraud'] = fraud_flags.values

    result = (
        temp[['velocity_fraud']]
        .reindex(original_index)
        ['velocity_fraud']
    )

    return result
    #Add Fraud Column
    df['velocity_fraud'] = flag_velocity_fraud(
    df,
    window='1h',
    min_txns=3
)

print(df['velocity_fraud'].value_counts())
#Compare Fraud vs Normal Transactions
comparison = (
    df.groupby('velocity_fraud')
      .agg(
          transaction_count=('amount', 'count'),
          avg_spend=('amount', 'mean'),
          decline_rate=('status',
                        lambda x: (x == 'declined').mean())
      )
)

print(comparison)
#Top Categories
top_categories = (
    df[df['velocity_fraud']]
    ['category']
    .value_counts()
    .head(5)
)

print("\nTop Categories in Fraud Transactions:")
print(top_categories)

#Summary Table
summary = pd.DataFrame({
    "Metric": [
        "Fraud Transactions",
        "Normal Transactions",
        "Fraud Rate (%)",
        "Avg Fraud Amount",
        "Avg Normal Amount"
    ],
    "Value": [
        df['velocity_fraud'].sum(),
        (~df['velocity_fraud']).sum(),
        100 * df['velocity_fraud'].mean(),
        df.loc[df['velocity_fraud'], 'amount'].mean(),
        df.loc[~df['velocity_fraud'], 'amount'].mean()
    ]
})

print(summary)

#Timeline Scatter Plot
import matplotlib.pyplot as plt

if 'datetime' not in df.columns:
    df['datetime'] = pd.to_datetime(
        df['date'].astype(str) + ' ' + df['time'].astype(str)
    )

users = df['user_id'].drop_duplicates()[:20]

plot_df = df[df['user_id'].isin(users)]

plt.figure(figsize=(12,6))

fraud = plot_df[plot_df['velocity_fraud']]
normal = plot_df[~plot_df['velocity_fraud']]

plt.scatter(
    normal['datetime'],
    normal['user_id'],
    label='Normal',
    alpha=0.6
)

plt.scatter(
    fraud['datetime'],
    fraud['user_id'],
    label='Velocity Fraud',
    alpha=0.8
)

plt.xlabel("Datetime")
plt.ylabel("User ID")
plt.title("Velocity Fraud Timeline")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

